# pyretechnics transmission prototype — North Cottonwood (COSWAP)

Step-by-step build of a paired counterfactual spread campaign with `fb_tools` + `pyretechnics`:
de-stack the LCP arms, pick real weather episodes, assemble hourly cubes, run exactly-paired
realizations, then read arrival time and firebrand production off the outputs at the treatment
units and the Strategic Focus Area.

Runs in the **`fb_pyre`** env. Decisions and gotchas: `docs/pyre_decisions.md`.

In [ ]:
import json, os, time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.plot

import fb_tools
from fb_tools.models import pyre
from fb_tools.spread import (destack_lcp, select_episodes, load_rtma_hourly, episode_hourly_frame,
                             weather_arrays, write_episode_frames, summarize_realization,
                             summarize_runs, paired_deltas, aggregate_realizations,
                             delta_burn_probability, check_domain_adequacy, weather_from_scenario)

# ── paths ────────────────────────────────────────────────────────────────────
COSWAP  = Path("/Users/mcc/Library/CloudStorage/Box-Box/CFRI/All Projects/COSWAP"
               "/Focus_Area_analysis/Analysis/fire_behavior")
LCP_DIR = COSWAP / "data/spatial/mod/transmission/lcp/North_Cottonwood"
IGN_DIR = COSWAP / "data/spatial/mod/transmission/ignitions"      # Phase 2 design fires
SFA_FP  = COSWAP / "data/spatial/raw/boundaries/Strategic_Focus_Areas_COSWAP.shp"
TRT_FP  = COSWAP / "data/spatial/mod/treatments/CFT_SFA_treatment_interactions.gpkg"

REPO     = Path(fb_tools.__file__).resolve().parents[1]
ERC_DIR  = REPO / "data/weather/pyrome_erc"
RTMA_DIR = REPO / "data/weather/pyrome_emc/RTMA_EMC"
WIND_DIR = None   # -> REPO / "data/weather/pyrome_emc/RTMA_WIND" once the 00b wind export is on disk
SCRATCH  = Path(os.environ.get("FB_PYRE_SCRATCH", Path.home() / "fb_scratch/coswap_pyre"))

# ── design ───────────────────────────────────────────────────────────────────
PYROME        = 45
ARMS          = ["untreated", "coswap"]          # add "background" for the 3-arm additivity check
BASELINE, TREATED = "untreated", "coswap"
SEVERITY      = "p97"
N_EPISODES    = 5
DURATION_DAYS = 21
CROWN_GATE    = 0.0                              # production choice; 0.4 = stock sensitivity
BASE_SEED     = 617327
N_WORKERS     = 4

pyre.pyre_version_info()

## 1. Static layers — de-stack each arm once

`destack_lcp` writes the eight pyretechnics inputs in engine units (slope as rise/run, canopy as
fractions/metres). Both arms are written on the same grid, so a pixel is the same pixel in every arm.

In [ ]:
layer_dirs = {arm: SCRATCH / "layers/North_Cottonwood" / arm for arm in ARMS}
for arm, d in layer_dirs.items():
    if not (d / "fuel_model.tif").exists():
        destack_lcp(LCP_DIR / f"{arm}.tif", d)

with rasterio.open(layer_dirs[BASELINE] / "fuel_model.tif") as src:
    GRID = dict(transform=src.transform, crs=src.crs, shape=src.shape, res=src.res[0])
    print(src.tags())
GRID["shape"], GRID["crs"]

## 1b. Single-run product: potential fire behavior under one scenario

No spread, no ignition, no comparison arm: `burn_landscape` burns every cell as a head fire under
one weather state (the engine's `burn_all_cells_as_head_fire`), giving the FlamMap-style map that
risk assessments and treatment prioritization start from. The weather comes straight from the FlamMap
scenario cache via `weather_from_scenario`, so this map is parameterized exactly like the in-situ
FlamMap runs beside it. Run it on any single LCP; the treated arm is optional.

In [ ]:
p97 = json.load(open(REPO / f"data/weather/flammap_rtma/pyrome_{PYROME}_flammap.json"))["scenarios"]["p97"]
manifest = json.load(open(COSWAP / "data/spatial/mod/transmission/run_manifest.json"))
WIND_FROM = manifest["wind_from_deg"]
scen = weather_from_scenario(p97, wind_from_deg=WIND_FROM, tmp_c=30.0)

base_map = {arm: pyre.burn_landscape(layer_dirs[arm], scen, SCRATCH / "potential" / SEVERITY / arm,
                                     crown_gate=CROWN_GATE, meta={"arm": arm, "severity": SEVERITY})
            for arm in ARMS}
pd.DataFrame(base_map).T[["n_burnable", "crown_frac", "wall_s", "crown_gate"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, layer, kw in zip(axes, ["flame_length", "fire_type"],
                         [dict(cmap="inferno", vmin=0, vmax=8), dict(cmap="viridis", vmin=0, vmax=3)]):
    with rasterio.open(SCRATCH / "potential" / SEVERITY / BASELINE / f"{layer}.tif") as src:
        im = ax.imshow(src.read(1), extent=rasterio.plot.plotting_extent(src), **kw)
    plt.colorbar(im, ax=ax, shrink=0.6, label=layer); ax.set_axis_off()
plt.suptitle(f"{BASELINE} @ {SEVERITY}: potential fire behavior (no spread)"); plt.show()

**Ember receptivity** is a second single-run product from the engine's exported spot-fire functions:
Schroeder ignition probability × flight survival. Under one scenario it is a scalar per landscape
(temperature and 1-hr FM are pyrome-mean here); with a per-pixel FM or temperature surface it becomes
a map. The docs' parameter exploration (§7.2.2) is reproduced below to show what the spotting
parameters actually control before any of them is swept.

In [ ]:
import pyretechnics.spot_fire as spot
from fb_tools.spread.pyre_outputs import firebrand_production

fm1_p97 = p97["FM_1hr"] / 100.0
print("P(ignition | ember lands) at 30 °C, p97 1-hr FM:", round(float(pyre.spot_ignition_probability(30.0, fm1_p97)), 3))
dist = np.arange(0, 2001, 100)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for dd in (250, 500, 1000):
    axes[0].plot(dist, pyre.spot_ignition_probability(30.0, fm1_p97, distance_m=dist, decay_distance=dd), label=f"decay {dd} m")
axes[0].set(xlabel="flight distance (m)", ylabel="P(spot ignition)"); axes[0].legend()
fm = np.linspace(0.02, 0.25, 50)
for tc in (20, 30, 40):
    axes[1].plot(fm * 100, [spot.schroeder_ignition_probability(tc, f) for f in fm], label=f"{tc} °C")
axes[1].set(xlabel="1-hr FM (%)", ylabel="Schroeder P(ignition)"); axes[1].legend()
I = np.logspace(1, 4.5, 60)
for flux in (1e-11, 5e-11, 2e-10):
    axes[2].plot(I, firebrand_production(I, 10.0, GRID["res"] ** 2, firebrands_per_unit_heat=flux), label=f"flux {flux:g}")
axes[2].set(xscale="log", yscale="log", xlabel="fireline intensity (kW/m) @ R = 10 m/min", ylabel="firebrands / cell"); axes[2].legend()
plt.tight_layout(); plt.show()

## 2. Weather episodes — real historical windows at the target severity

An episode is a named 21-day window whose day-1 ERC is at or above the pyrome's `p97`. Nothing is
scaled: the diurnal cycle, fronts, and recovery come straight from the record. The RTMA record is
loaded first (dead FM spun up over every season) so selection can skip windows with export gaps —
2012–2015 and 2024 have day-scale holes; 2016–2023 and 2025 are complete.

In [ ]:
rtma = load_rtma_hourly(RTMA_DIR, PYROME, range(2011, 2026), wind_dir=WIND_DIR)
episodes = select_episodes(PYROME, ERC_DIR, severity=SEVERITY, n=N_EPISODES, duration_days=DURATION_DAYS,
                           rtma_df=rtma, seed=BASE_SEED)
episodes

In [ ]:
from fb_tools.weather.gridmet import load_gridmet_pyrome_cache
erc = np.asarray(load_gridmet_pyrome_cache(PYROME, ERC_DIR, return_meta=True)["HistoricERCValues"])
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(erc.ravel(), bins=40, color="0.8")
ax.axvline(episodes.erc_threshold[0], color="k", ls="--", label=SEVERITY)
for _, e in episodes.iterrows():
    ax.axvline(e.erc_day1, color="C3", alpha=0.6)
ax.set(xlabel="ERC (all season-days, 2010-2025)", ylabel="days"); ax.legend(); plt.show()

## 3. Hourly driver frames

Dead FM from the hourly RTMA EMC record via the NFDRS78 lag (spun up over the whole season);
live FM scaled from FM100; temperature for Schroeder ignition probability.

**Wind**: until the RTMA wind export (`00b_RTMA-EMC.ipynb`, last cells) is on disk, the frame
uses a constant p97 wind from the FlamMap scenario cache. That is a placeholder — spotting rides
the wind tail, and a constant wind has none.

In [ ]:
WIND = None if WIND_DIR else (float(scen["wind_speed_10m"][0]), WIND_FROM)   # constant p97 placeholder
print("wind:", "RTMA hourly" if WIND is None else f"constant {WIND[0]:.1f} km/h from {WIND[1]}°")

episode_csvs = write_episode_frames(episodes, rtma, SCRATCH / "episodes", wind=WIND)
frames = {k: pd.read_csv(v) for k, v in episode_csvs.items()}
frames["ep_000"].head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for ep, f in frames.items():
    axes[0].plot(f.hour / 24, f.tmp_c, lw=0.8, label=ep)
    axes[1].plot(f.hour / 24, f.fm1 * 100, lw=0.8)
    axes[2].plot(f.hour / 24, f.ws_kmh, lw=0.8)
axes[0].set_ylabel("T (°C)"); axes[1].set_ylabel("1-hr FM (%)"); axes[2].set_ylabel("wind (km/h)")
axes[2].set_xlabel("day"); axes[0].legend(ncol=5, fontsize=7); plt.tight_layout(); plt.show()

## 4. Reporting zones

Treatment units (dissolved), a 500 m collar around them, and the SFA polygon covering the domain.
Every per-realization metric is read inside each zone: first arrival hour, fraction burned, mean
fireline intensity, expected firebrands produced.

In [ ]:
from shapely.geometry import box
domain = box(*rasterio.transform.array_bounds(*GRID["shape"], GRID["transform"]))   # (w, s, e, n)

trt = gpd.read_file(TRT_FP).to_crs(GRID["crs"]).clip(domain)
sfa = gpd.read_file(SFA_FP).to_crs(GRID["crs"]).clip(domain)
print("treatment columns:", list(trt.columns)); print("SFA columns:", list(sfa.columns))

trt_u = trt.geometry.union_all()
zones = gpd.GeoDataFrame({"zone": ["trt", "collar_500m", "sfa"],
                          "geometry": [trt_u, trt_u.buffer(500).difference(trt_u), sfa.geometry.union_all()]},
                         crs=GRID["crs"])
zones["area_ha"] = zones.area / 1e4
zones

## 5. One realization, inspected

Baseline arm, first design fire, first episode. Spotting on at pyretechnics' central flux; crown gate
at the production setting. Outputs are five rasters (arrival minutes, fire type, spread rate,
intensity, flame length) plus a manifest that records the seed, engine build, gate and solver knobs.

In [ ]:
ignitions = {p.stem: p for p in sorted(IGN_DIR.glob("ign_*.shp"))}
ign0, ep0 = next(iter(ignitions)), "ep_000"
smoke = SCRATCH / "smoke" / BASELINE

t0 = time.perf_counter()
man = pyre.run_realization(layer_dirs[BASELINE], weather_arrays(frames[ep0]), ignitions[ign0], smoke,
                           seed=pyre.seed_for(BASE_SEED, ign0, ep0, "central"),
                           duration_h=DURATION_DAYS * 24, spot_config=pyre.SPOT_CENTRAL,
                           crown_gate=CROWN_GATE, overwrite=True)
print(f"{time.perf_counter() - t0:.0f} s  burned {man['burned_ha']:,.0f} ha  {man['stop_condition']}")
{k: man[k] for k in ("seed", "crown_gate", "pyretechnics_version", "pending_spot_cells")}

In [ ]:
with rasterio.open(smoke / "time_of_arrival.tif") as src:
    toa_d = src.read(1) / 1440.0
    extent = rasterio.plot.plotting_extent(src)
fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(toa_d, extent=extent, cmap="magma_r", vmin=0, vmax=DURATION_DAYS)
zones.boundary.plot(ax=ax, color=["C0", "C0", "C2"], lw=0.8)
gpd.read_file(ignitions[ign0]).to_crs(GRID["crs"]).plot(ax=ax, color="cyan")
plt.colorbar(im, ax=ax, shrink=0.7, label="arrival (days)"); ax.set_axis_off(); plt.show()

pd.Series(summarize_realization(smoke, zones=zones))

## 6. The paired campaign

One row per `(ignition, episode, condition) × arm`. The seed derives from the realization key and
never from the arm, and the ignition is rasterized once and shared — so the two arms of a realization
differ **only** in the fuel layers. `assert_crn_contract` checks the table before anything runs.

In [ ]:
conditions = {"central": {"spot_config": pyre.SPOT_CENTRAL, "crown_gate": CROWN_GATE}}
runs = pyre.build_runs({a: layer_dirs[a] for a in ARMS}, ignitions, episode_csvs, conditions,
                       base_seed=BASE_SEED, out_root=SCRATCH / "runs")
runs.to_csv(SCRATCH / "runs.csv", index=False)
print(len(runs), "realizations")
runs.head()

In [ ]:
# ~4-5 min per realization at 30 m / 21 d with spotting; finished rows are skipped on re-run.
res = pyre.run_batch(runs, duration_h=DURATION_DAYS * 24, n_workers=N_WORKERS)
res.to_csv(SCRATCH / "results.csv", index=False)
res.groupby("arm").burned_ha.describe()

## 7. Paired effects: arrival time and firebrand production

`paired_deltas` is treated minus baseline on the shared realization key. Positive arrival delta =
the treatment delayed the fire. Firebrands are pyretechnics' expected production
(`f · 60 I / R · cell area`) summed over each zone.

Spotting is a heavy tail (Stage 2): report per-ignition effects and a breakout rate, not a mean alone.

In [ ]:
summ = summarize_runs(res, zones=zones)
summ.to_csv(SCRATCH / "summary.csv", index=False)
d = paired_deltas(summ, baseline=BASELINE, treated=TREATED)

cols = ["d_burned_ha", "d_area_d3_ha", "d_trt_arrival_h", "d_collar_500m_arrival_h",
        "d_sfa_arrival_h", "d_trt_firebrands", "d_sfa_firebrands", "d_trt_fi_mean"]
d[["ignition_id", "episode_id"] + cols].round(1)

In [ ]:
pooled = d[cols].agg(["mean", "median", "std", "count"]).T
per_ign = d.groupby("ignition_id")[cols].mean().T
pd.concat({"pooled": pooled, "per_ignition": per_ign}, axis=1).round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, c in zip(axes, ["d_sfa_arrival_h", "d_sfa_firebrands", "d_burned_ha"]):
    d.boxplot(column=c, by="ignition_id", ax=ax); ax.axhline(0, color="k", lw=0.6); ax.set_title(c)
plt.suptitle(f"{TREATED} − {BASELINE}, {len(d)} paired realizations"); plt.tight_layout(); plt.show()

## 8. Ensemble rasters and ΔBP

`aggregate_realizations` collapses each arm into `BurnProb / AvgFlameLength / AvgTime(days)` — the
rasters `bp.py` already consumes, so `delta_burn_probability` and the zonal summaries work unchanged.
`check_domain_adequacy` is the Stage 2 open gate: the 52 km domain failed it under constant p97 wind.

In [ ]:
ens = {arm: aggregate_realizations(res[(res.arm == arm) & (res.status == "ok")].out_dir,
                                   SCRATCH / "ensemble" / arm) for arm in ARMS}
dbp = delta_burn_probability(ens[BASELINE]["BurnProb"], ens[TREATED]["BurnProb"],
                             out_path=SCRATCH / "ensemble" / f"dBP_{TREATED}_{BASELINE}.tif")
adequacy = check_domain_adequacy(ens[BASELINE]["BurnProb"])
{k: adequacy[k] for k in adequacy if k.endswith("_ok")}

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
with rasterio.open(SCRATCH / "ensemble" / f"dBP_{TREATED}_{BASELINE}.tif") as src:
    a = src.read(1, masked=True) / 100.0
    im = ax.imshow(a, extent=rasterio.plot.plotting_extent(src), cmap="RdBu", vmin=-0.3, vmax=0.3)
zones.boundary.plot(ax=ax, color="k", lw=0.6)
plt.colorbar(im, ax=ax, shrink=0.7, label=f"ΔBP ({TREATED} − {BASELINE})"); ax.set_axis_off(); plt.show()

## Next

- Replace the constant wind with the RTMA hourly wind export, then re-run — the campaign table and
  seeds are unchanged, so results are directly comparable.
- Add `background` as a third arm for the additivity check; add `"off"` and flux `low`/`high`
  conditions for the spotting sweep.
- Realized spot-ignition cells are not exposed by the engine's final matrices; recover them by
  stepping the run in chunks and diffing `spot_ignitions` (decision log, open item).